# Module 9 — Coverage & CI (code-along)

You have a green suite (M7–M8). Two questions remain:
1. **How much** of your code does it actually run? — *coverage*
2. Does it run **without you remembering** to? — *CI*

> **This code-along is a terminal exercise.** Coverage and CI measure *real files* and *real CI
> runs* — neither happens inside a notebook. The one code cell below shows the *idea* of an
> untested branch; everything else you run in a terminal on your own repo (marked **▶ Terminal
> moment**).

## §1 · Coverage — green ≠ tested

A suite can pass while whole branches are **never executed**. "All green" means the tests
you *wrote* pass — not that your code is *covered*. Below: a two-branch function whose test
only exercises one branch.

In [ ]:
def apply_discount(price, is_member):
    if is_member:
        return round(price * 0.9, 2)   # branch A
    return price                       # branch B  <-- never run below

# our only "test" checks the member branch...
assert apply_discount(100, True) == 90.0
print("test passed ✓  — but branch B (non-member) was never executed")
print("coverage would list `return price` in the Missing column")

### ▶ Terminal moment — coverage on *real* code

The cell above shows the *idea*; the number only means something on real files. Run it on your
catalog now:

```bash
uv run pytest --cov=catalog --cov-report=term-missing --cov-branch
```

Read the **Missing** column — those are the lines no test reached. Then add `--cov-fail-under=80`
and watch the build **fail** when coverage dips below the line. That's coverage becoming a gate.

### Measure it — `pytest-cov`

`pytest --cov` reports how much of each file ran; `--cov-report=term-missing` names the
**exact lines** no test reached. On the Day-3 catalog suite you'd see something like:

```
Name                 Stmts   Miss Branch BrPart  Cover   Missing
----------------------------------------------------------------
catalog/models.py       74      0     12      0   100%
catalog/server.py       38      8      0      0    79%   46-49, 54-57
catalog/storage.py      43     18      6      0    55%   39-47, 51-59
----------------------------------------------------------------
TOTAL                  155     26     18      0    83%
```

`branch = true` counts *branch* coverage (was each `if` taken both ways?), not just lines.
83% isn't a failure — the *Missing* column found real gaps: the `PATCH`/`DELETE` routes and
the CSV helpers your tests never touched. **Coverage is a floor, not a target:** a line that
*ran* can still be *wrong* — use the number to find the untested path that matters.

## §2 · CI — run it on every push (GitHub Actions)

A green suite on *your* laptop protects nothing when a teammate pushes a breaking change and
never runs it. **CI** runs the suite automatically on every push and pull request. A YAML
file in `.github/workflows/` describes it:

```yaml
on: [push, pull_request, workflow_dispatch]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false                       # one version failing must not cancel the others
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5        # this project is uv-based (dev deps in [dependency-groups])
      - uses: actions/setup-python@v5
        with: { python-version: ${{ matrix.python-version }} }
      - run: uv sync                       # installs pytest/pytest-cov/httpx from the dev group
      - run: uv run pytest --cov --cov-report=xml --html=report.html --self-contained-html
      - uses: actions/upload-artifact@v4
        if: always()                         # upload even when tests FAIL — you need it most then
        with: { name: report, path: report.html }
```

- **matrix** runs the suite across three Python versions at once.
- **`if: always()`** keeps the report from a *failing* run.
- add **`--cov-fail-under=80`** to turn coverage into a gate: red can't merge.

---
**Now do the lab** — `modules/m09-coverage-ci/lab/README.md`:
fill the coverage config in `pyproject.toml` and every `# TODO` in
`.github/workflows/tests.yml`. Run `uv run pytest --cov --cov-report=term-missing -q`
→ a coverage table + **31 passed**.

**End of Day 3** — your catalog is tested (M7), its network edge mocked (M8), and the whole
suite measured and automated (M9).